# Model B: Duplicate Identity Detection (Transliteration Handling)

Exact string matching fails on Gujarati-to-English transliterations (e.g., *Suresh Patel* vs *Sureshbhai Ptl*).

In this notebook, we load the real `TS-PS4-1.csv` dataset, filter for a specific district to optimize computation, and run **Levenshtein Distance** algorithms (`FuzzyWuzzy`) to flag duplicate identities.


## 1. Load Data and Filter District


In [3]:
# !pip install fuzzywuzzy python-Levenshtein
from fuzzywuzzy import fuzz
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load data and get unique beneficiaries (we only need one record per beneficiary)
df = pd.read_csv('../data/TS-PS4-1.csv')
unique_ben = df.drop_duplicates(subset=['beneficiary_id'])

# Filter for 'Surat' district to demonstrate the concept quickly
surat_df = unique_ben[unique_ben['district'] == 'Surat'].reset_index(drop=True)
print(f"Loaded {len(surat_df)} unique beneficiaries in Surat.")

Loaded 9973 unique beneficiaries in Surat.


## 2. Execute Fuzzy Matching Engine


In [4]:
results = []

# We compare names within the same district. 
# In a production environment with millions of rows, we would use Phonetic Hashing 
# (Double Metaphone) as a blocking key before running computationally expensive fuzzy matches.
# Here we demonstrate the O(N^2) comparison on a small subset (e.g., first 100 records in Surat).

subset = surat_df.head(100)

for i in range(len(subset)):
    for j in range(i + 1, len(subset)):
        name1 = str(subset.iloc[i]['name']).lower()
        name2 = str(subset.iloc[j]['name']).lower()
        
        # Calculate similarity ratios
        score = fuzz.ratio(name1, name2)
        token_score = fuzz.token_sort_ratio(name1, name2)
        final_score = max(score, token_score)
        
        # Flag if similarity is high but Aadhaar/Beneficiary IDs differ
        if final_score > 85 and final_score < 100: 
            results.append({
                'beneficiary_a': subset.iloc[i]['beneficiary_id'],
                'name_a': subset.iloc[i]['name'],
                'beneficiary_b': subset.iloc[j]['beneficiary_id'],
                'name_b': subset.iloc[j]['name'],
                'similarity_score': final_score
            })

df_results = pd.DataFrame(results)

# Sort by similarity
if not df_results.empty:
    df_results = df_results.sort_values(by='similarity_score', ascending=False)
    print("--- Transliteration Matches Flagged in Surat ---")
    print(df_results.head(10))
else:
    print("No immediate transliteration duplicates found in this subset.")


--- Transliteration Matches Flagged in Surat ---
    beneficiary_a        name_a beneficiary_b        name_b  similarity_score
0         B100000    Suresh Ptl       B100047  Suresh Patel                91
1         B100000    Suresh Ptl       B100072  Suresh Patel                91
94        B100121    Suresh Ptl       B100465  Suresh Patel                91
95        B100121    Suresh Ptl       B100502  Suresh Patel                91
96        B100121    Suresh Ptl       B100521  Suresh Patel                91
148       B100271  Ramesh Patel       B100447    Rmsh Patel                91
146       B100271  Ramesh Patel       B100296    Rmsh Patel                91
145       B100261    Rmsh Patel       B100541  Ramesh Patel                91
144       B100261    Rmsh Patel       B100537  Ramesh Patel                91
177       B100356  Ramesh Patel       B100447    Rmsh Patel                91
